# EDEN-06 — Vision recovery and feature extraction | v1.0

**Phase 2: Visual intelligence & climate baselines.** This notebook completes the implementation of the original vision stages 04, 05 and 06 together: simple trait predictors → RGB/depth/fused models → frozen image features. The existing climate notebooks called 04/05 are earlier climate prototypes, not completion of the original vision stages.

**Kaggle:** select a GPU (one T4 is enough), Internet **On**, and attach the EDEN-01 v1.1, EDEN-02 v1.1 and EDEN-03 v1.0 result bundles (ZIPs or extracted folders). Climate 04/05 results are not inputs. Run all cells in order. This notebook has not been executed locally.

**Extra data:** automatically read 383 color/depth pairs and metadata from the public 2023 labeled pre-trial, about **5.61 GB** of compressed image data, instead of the whole 7.46 GB archive. Publisher test images/labels and infrared images are excluded. Keep Internet on for data and pretrained weights. The first acquisition can take a while; completed resized pairs are cached in this Kaggle session. The cache is not included in the result ZIP.

**What you get:** validation baselines and ablations, a frozen RGB-D model, 2023 out-of-fold features, and 2024 canopy features ready for original Notebook 07 alignment. A feature is a number describing an image. Out-of-fold means that a training example's prediction comes from a model that did not train on its plant group.

**Honest limits:** the pre-trial has single-plant labels; the 2024 pilot has canopy photos without matching trait labels. Canopy trait predictions are explicitly **unvalidated transfer estimates**, not measured canopy height/weight. Depth stays relative because its physical unit is unverified. Empty color masks and dark frames remain flagged. No segmentation accuracy is claimed without human-drawn masks. An implementation pass does not mean the models are accurate enough for deployment.

Sources: [original labeled pre-trial](https://doi.org/10.4121/e1ee9de9-6ce9-4502-a37c-34b5b1372bed.v1), [2024 challenge](https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1), CC BY 4.0. Design: EDEN SRS Implementation Handbook, original modules 04–06. API references: [MobileNetV3](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html), [group splitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html). A preparation-time source inspection included test metadata; no test scores or test-driven model selection were performed. This notebook never loads publisher test labels.

## Cell 1 — Settings and folders

The settings below control the full run. GPU training includes three comparisons and three out-of-fold fits. Nothing silently switches to a smaller demonstration run.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import gc, hashlib, io, json, random, re, time, urllib.request, zipfile
import importlib.metadata
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ROOT = WORK / 'eden06'
CACHE = WORK / 'eden06_cache'
RESULT = WORK / 'EDEN-06-Vision-Recovery-and-Features-Results-v1.0.zip'
SEED, IMAGE_SIZE, BATCH_SIZE = 42, 224, 16
MAX_EPOCHS, PATIENCE, N_FOLDS, MC_PASSES = 40, 7, 3, 20
LR, WEIGHT_DECAY = 3e-4, 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Enable a Kaggle GPU before running this training notebook.'
# Reuse only validated caches/checkpoints; output files are rebuilt without deleting other folders.
for directory in [ROOT / 'reports', ROOT / 'vision', ROOT / 'models', ROOT / 'data', CACHE]:
    directory.mkdir(parents=True, exist_ok=True)

def sha(payload):
    """Return a fingerprint so changed input bytes cannot pass unnoticed."""
    return hashlib.sha256(payload).hexdigest()

def save_json(path, value):
    """Write readable metadata; reject non-standard NaN values in reports."""
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')

def seed_all(seed):
    """Repeat data ordering and initialization; GPU results can still vary slightly."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_all(SEED)
VERSIONS = {p: importlib.metadata.version(p) for p in
            ['numpy', 'pandas', 'Pillow', 'torch', 'torchvision', 'scikit-learn', 'pyarrow']}
save_json(ROOT / 'reports/runtime.json', VERSIONS)
print('Device:', DEVICE, '| full training:', MAX_EPOCHS, 'epochs maximum per comparison')

## Cell 2 — Verify the three upstream results

A manifest is a receipt listing files and their fingerprints. This cell checks every listed file, the frozen splits, and which upstream run produced each result.

In [ ]:
class Source:
    def __init__(self, path, member=None):
        self.path, self.member = Path(path), member
        if member is None:
            self.manifest_bytes = self.path.read_bytes()
            self.base = self.path.parent
        else:
            with zipfile.ZipFile(self.path) as archive:
                self.manifest_bytes = archive.read(member)
            self.prefix = str(PurePosixPath(member).parent)
        self.manifest = json.loads(self.manifest_bytes)

    def read(self, relative):
        # Accept only paths relative to the producer's artifact directory.
        relative = str(relative).replace(chr(92), '/')
        parts = PurePosixPath(relative)
        assert not parts.is_absolute() and '..' not in parts.parts and ':' not in relative
        if self.member is None:
            target = (self.base / relative).resolve()
            assert target.is_relative_to(self.base.resolve())
            return target.read_bytes()
        name = str(PurePosixPath(self.prefix) / parts)
        with zipfile.ZipFile(self.path) as archive:
            return archive.read(name)


needed = {'EDEN-01': ('1.1', 'PASS_ETL'), 'EDEN-02': ('1.1', 'PASS_RGBD_PILOT'),
          'EDEN-03': ('1.0', 'PARTIAL_PASS_QC_FLAGS_PRESENT')}
candidates = {name: {} for name in needed}

def consider(source):
    """Keep distinct manifests for only the expected producers and versions."""
    producer = source.manifest.get('producer')
    if producer in needed and str(source.manifest.get('code_version')) == needed[producer][0]:
        candidates[producer][sha(source.manifest_bytes)] = source

for path in sorted(INPUT.rglob('*.zip')):
    try:
        with zipfile.ZipFile(path) as archive:
            members = [n for n in archive.namelist() if PurePosixPath(n).name == 'manifest.json']
        for member in members:
            consider(Source(path, member))
    except (zipfile.BadZipFile, json.JSONDecodeError, UnicodeDecodeError):
        continue  # An unrelated attachment is not an EDEN result.
for path in sorted(INPUT.rglob('manifest.json')):
    try:
        consider(Source(path))
    except (json.JSONDecodeError, UnicodeDecodeError):
        continue
sources, verified = {}, []
for producer, (_, status) in needed.items():
    assert len(candidates[producer]) == 1, f'Attach exactly one distinct {producer} result.'
    source = next(iter(candidates[producer].values()))
    assert source.manifest['status'] == status, f'Unexpected upstream status: {producer}'
    files = source.manifest['files']
    assert files and len({f['path'] for f in files}) == len(files)
    for item in files:
        payload = source.read(item['path'])
        assert len(payload) == item['size'] and sha(payload) == item['sha256'], item['path']
    sources[producer] = source
    verified.append(dict(producer=producer, files=len(files), manifest_sha256=sha(source.manifest_bytes)))
    print('Verified', producer, len(files), 'files')
s01, s02, s03 = [sources[p] for p in needed]
assert s02.manifest['upstream_manifest_sha256'] == sha(s01.manifest_bytes)
assert s03.manifest['input_manifest_sha256'] == sha(s02.manifest_bytes)
assert s01.read('data/splits.json') == s02.read('data/splits.json')
(ROOT / 'data/splits.json').write_bytes(s01.read('data/splits.json'))
images = pd.read_csv(io.BytesIO(s02.read('vision/images.csv')), dtype={'compartment_id': str})
spatial = pd.read_csv(io.BytesIO(s03.read('vision/spatial_features.csv')), dtype={'compartment_id': str})
assert images.image_id.is_unique and set(images.image_id) == set(spatial.image_id)
images['ts_utc'] = pd.to_datetime(images.ts_utc, utc=True)
save_json(ROOT / 'reports/input_verification.json', verified)

## Cell 3 — Open the labeled pre-trial without downloading the archive

HTTP Range asks the server for just the bytes we need. ZIP CRC checks catch damaged individual files. This is partial-file verification, not a claim that we checked the whole archive MD5.

In [ ]:
ARTICLE = 'e1ee9de9-6ce9-4502-a37c-34b5b1372bed'
FILE_ID = '923e3ab0-011f-432c-9039-2151e2c4a6f9'
ARCHIVE_NAME = '4th_autonomous_greenhouse_challenge_dwarf_tomato_pretrial_single_plant_cam.zip'
ARCHIVE_SIZE = 7456535483
URL = f'https://data.4tu.nl/file/{ARTICLE}/{FILE_ID}'
NETWORK_BUDGET = 8 * 1024**3

class RangeReader(io.RawIOBase):
    """Expose a remote ZIP as a seekable file with strict, bounded range requests."""
    def __init__(self, url, size):
        self.url, self.size, self.pos = url, size, 0
        self.transferred, self.cache_start, self.cache = 0, 0, b''

    def seekable(self):
        return True

    def seek(self, offset, whence=0):
        self.pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset
        if not 0 <= self.pos <= self.size:
            raise ValueError('Range seek outside the source file')
        return self.pos

    def tell(self):
        return self.pos

    def fetch(self, start, count):
        """Fetch one bounded chunk; reject servers that send the whole file."""
        count = min(count, self.size - start)
        assert 0 < count <= 64 * 1024**2
        for attempt in range(3):
            assert self.transferred + count <= NETWORK_BUDGET, 'Network budget reached; keep cache and retry deliberately.'
            self.transferred += count  # Count retries conservatively against the budget.
            request = urllib.request.Request(self.url, headers={
                'Range': f'bytes={start}-{start + count - 1}', 'Accept-Encoding': 'identity'})
            try:
                with urllib.request.urlopen(request, timeout=120) as response:
                    assert response.status == 206, 'Server ignored Range; full download refused.'
                    assert response.headers.get('Content-Range') == f'bytes {start}-{start + count - 1}/{self.size}'
                    payload = response.read(count + 1)
                    assert len(payload) == count, 'Truncated range response'
                self.cache_start, self.cache = start, payload
                return
            except (OSError, TimeoutError):
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)

    def read(self, count=-1):
        count = self.size - self.pos if count < 0 else min(count, self.size - self.pos)
        if not count:
            return b''
        if not (self.cache_start <= self.pos and self.pos + count <= self.cache_start + len(self.cache)):
            self.fetch(self.pos, max(count, min(65536, self.size - self.pos)))
        offset = self.pos - self.cache_start
        self.pos += count
        return self.cache[offset:offset + count]

    def warm_member(self, info):
        """Read a whole compressed member plus its small header in one request."""
        self.fetch(info.header_offset, min(info.compress_size + 65536, self.size - info.header_offset))

attached = list(INPUT.rglob(ARCHIVE_NAME))
assert len(attached) <= 1, 'Detach duplicate pre-trial archives.'
remote = None if attached else RangeReader(URL, ARCHIVE_SIZE)
if attached:
    assert attached[0].stat().st_size == ARCHIVE_SIZE
archive = zipfile.ZipFile(attached[0] if attached else remote)
members = {i.filename: i for i in archive.infolist() if not i.is_dir() and '__MACOSX' not in i.filename}

def read_member(name):
    """Read a real member; zipfile also verifies its stored CRC."""
    assert name in members, f'Source member missing: {name}'
    if remote is not None:
        remote.warm_member(members[name])
    return archive.read(name)

publisher_readme = read_member('single_plant_cam/ReadMe.md').decode('utf-8-sig')
(ROOT / 'reports/pretrial_ReadMe.md').write_text(publisher_readme, encoding='utf-8')
(ROOT / 'reports/pretrial_camera_config.json').write_bytes(read_member('single_plant_cam/oak-d-s2-poe_conf.json'))
print('Opened source index; publisher test and infrared files will not be read.')

## Cell 4 — Match images, labels, and plant groups

A label is a measured answer used to teach the model. We use height, fresh weight, leaf area, and red-fruit count where present. To keep repeated identities together, this experiment makes its own grouped development split from publisher train/validation data; publisher test stays excluded. These are separate 2023 splits and do not change the frozen 2024 splits.

In [ ]:
TARGETS = ['height_cm', 'fresh_weight_g', 'leaf_area_cm2', 'red_fruit_count']
SOURCE_FIELDS = ['height', 'fw_plant', 'leaf_area', 'number_of_red_fruits']
UNITS = ['cm', 'g', 'cm2', 'count']
rows, metadata_receipts = [], []
for role in ['train', 'validation1', 'validation2', 'validation3']:
    prefix = f'single_plant_cam/{role}/'
    label_name, key_name = prefix + f'ground_truth_{role}.json', prefix + f'anon_keys_{role}.json'
    label_bytes, key_bytes = read_member(label_name), read_member(key_name)
    labels, keys = json.loads(label_bytes), json.loads(key_bytes)
    metadata_receipts.extend([dict(path=label_name, sha256=sha(label_bytes)), dict(path=key_name, sha256=sha(key_bytes))])
    assert set(labels) == set(keys)
    for sample_id, values in labels.items():
        original = keys[sample_id]
        match = re.fullmatch(r'(.+?)_(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_([^_]+)_rgb', original)
        assert match, f'Unexpected publisher identifier: {original}'
        camera, local_time, plant_id = match.groups()
        # RGB is named ID.png; depth is ID_depth.png. Infrared files are separate.
        paths = {}
        for kind in ['rgb', 'depth']:
            filename = f'{sample_id}.png' if kind == 'rgb' else f'{sample_id}_depth.png'
            matches = [n for n in members if n.startswith(prefix + kind + '/') and PurePosixPath(n).name == filename]
            assert len(matches) == 1, (sample_id, kind, matches)
            paths[kind + '_member'] = matches[0]
        row = dict(sample_id=sample_id, publisher_role=role, original_name=original,
                   plant_group=plant_id, camera_id=camera,
                   ts_utc=pd.Timestamp(local_time.replace('_', '-', 2).replace('_', ' ', 1).replace('_', ':')).tz_localize('Europe/Amsterdam').tz_convert('UTC'), **paths)
        for target, field in zip(TARGETS, SOURCE_FIELDS):
            value = float(values.get(field, np.nan))
            row[target] = value if np.isfinite(value) and value >= 0 else np.nan
        rows.append(row)
pretrial = pd.DataFrame(rows).sort_values('sample_id').reset_index(drop=True)
assert len(pretrial) == 383 and pretrial.sample_id.is_unique, 'Pinned source inventory changed; review it.'
assert pretrial.plant_group.nunique() >= 12
train_indices, validation_indices = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(pretrial, groups=pretrial.plant_group))
pretrial['split'] = 'validation'
pretrial.loc[train_indices, 'split'] = 'train'
assert not set(pretrial.loc[train_indices, 'plant_group']) & set(pretrial.loc[validation_indices, 'plant_group'])
pretrial['fold'] = -1
for fold, (_, held) in enumerate(GroupKFold(N_FOLDS).split(train_indices, groups=pretrial.loc[train_indices, 'plant_group'])):
    pretrial.loc[train_indices[held], 'fold'] = fold
for indices in [train_indices, validation_indices]:
    assert (pretrial.loc[indices, TARGETS].notna().sum() >= 3).all(), 'Too few measured labels for a target.'
selected = [members[n] for column in ['rgb_member', 'depth_member'] for n in pretrial[column]]
selected_bytes = sum(i.compress_size for i in selected)
assert selected_bytes < NETWORK_BUDGET * 0.9
pretrial.to_parquet(ROOT / 'data/pretrial_split.parquet', index=False)
save_json(ROOT / 'reports/label_contract.json', dict(
    targets=TARGETS, source_fields=SOURCE_FIELDS, units=UNITS, seed=SEED, group='plant ID across all dates',
    split='custom 80/20 grouped development split; publisher test excluded',
    fresh_weight_note='Source fw_plant field; source README contains an inconsistent lettuce description. No plant-part claim is added.',
    depth_units='unverified; no conversion to physical distance', metadata=metadata_receipts))
display(pretrial.groupby('split')[TARGETS].count())
print('Selected compressed image bytes:', selected_bytes, '| pairs:', len(pretrial))

## Cell 5 — Cache paired inputs and simple image measurements

Color and depth receive exactly the same resizing and padding. Depth uses nearest-neighbor resizing so it does not invent intermediate codes. We divide nonzero depth by its own image median to obtain a relative shape signal, not centimetres. Green masks are rough color rules, not verified plant outlines.

In [ ]:
PREPROCESS_VERSION = 'paired-letterbox224-relative-depth-v1'

def prepare_pair(rgb_bytes, depth_bytes):
    """Decode a pair, measure simple features, and make synchronized model inputs."""
    rgb = np.asarray(Image.open(io.BytesIO(rgb_bytes)).convert('RGB'))
    depth = np.asarray(Image.open(io.BytesIO(depth_bytes)))
    # Pillow versions may decode 16-bit PNG as uint16 or int32; both retain the codes.
    assert depth.ndim == 2 and depth.shape == rgb.shape[:2] and depth.dtype.kind in 'ui'
    assert depth.min() >= 0 and depth.max() <= 65535
    height, width = depth.shape
    scale = IMAGE_SIZE / max(height, width)
    size = (max(1, round(width * scale)), max(1, round(height * scale)))
    small_rgb = np.asarray(Image.fromarray(rgb).resize(size, Image.Resampling.BILINEAR))
    small_depth = np.asarray(Image.fromarray(depth).resize(size, Image.Resampling.NEAREST)).astype(np.float32)
    # Zero is provisionally masked, while 65535 is recorded as a special-code flag.
    valid = small_depth > 0
    median = float(np.median(small_depth[valid])) if valid.any() else 1.0
    relative = np.where(valid, np.clip(small_depth / median, 0, 4), 0)
    rgb_input = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), np.uint8)
    depth_input = np.zeros((2, IMAGE_SIZE, IMAGE_SIZE), np.float32)
    x, y = (IMAGE_SIZE - size[0]) // 2, (IMAGE_SIZE - size[1]) // 2
    rgb_input[y:y + size[1], x:x + size[0]] = small_rgb
    depth_input[0, y:y + size[1], x:x + size[0]] = relative
    depth_input[1, y:y + size[1], x:x + size[0]] = valid
    color = small_rgb.astype(np.float32) / 255
    red, green, blue = color[..., 0], color[..., 1], color[..., 2]
    hsv = np.asarray(Image.fromarray(small_rgb).convert('HSV')).astype(float) / 255
    exg = 2 * green - red - blue
    mask = (hsv[..., 0] >= 0.15) & (hsv[..., 0] <= 0.48) & (hsv[..., 1] > 0.15) & (exg > 15 / 255)
    mask_depth = small_depth[mask & valid]
    percentiles = np.percentile(mask_depth, [5, 50, 95]) if mask_depth.size else [np.nan] * 3
    feature = dict(rgb_brightness=float(rgb.mean()), green_fraction=float(mask.mean()),
                   depth_nonzero_fraction=float((depth > 0).mean()), depth_65535_fraction=float((depth == 65535).mean()),
                   depth_code_p05=float(percentiles[0]), depth_code_p50=float(percentiles[1]), depth_code_p95=float(percentiles[2]))
    for channel, name in enumerate(['red', 'green', 'blue']):
        feature[f'{name}_mean'] = float(color[..., channel].mean())
        feature[f'{name}_std'] = float(color[..., channel].std())
    feature['very_dark'] = feature['rgb_brightness'] < 5
    feature['empty_green_mask'] = not bool(mask.any())
    return rgb_input, depth_input, feature

def cached_pair(sample_id, rgb_member, depth_member, read, identity):
    """Reuse a cache only when source identity, settings, and cached bytes agree."""
    fingerprint = sha(json.dumps([PREPROCESS_VERSION, IMAGE_SIZE, identity, rgb_member, depth_member]).encode())
    cache_path = CACHE / f'{sample_id}-{fingerprint[:16]}.npz'
    receipt_path = cache_path.with_suffix('.json')
    if cache_path.exists() and receipt_path.exists():
        receipt = json.loads(receipt_path.read_text())
        if receipt['cache_sha256'] == sha(cache_path.read_bytes()) and receipt['fingerprint'] == fingerprint:
            return cache_path, receipt
    rgb_bytes, depth_bytes = read(rgb_member), read(depth_member)
    rgb, depth, features = prepare_pair(rgb_bytes, depth_bytes)
    np.savez_compressed(cache_path, rgb=rgb, depth=depth.astype(np.float16))
    receipt = dict(fingerprint=fingerprint, cache_sha256=sha(cache_path.read_bytes()),
                   rgb_sha256=sha(rgb_bytes), depth_sha256=sha(depth_bytes), features=features)
    # Missing mask-depth percentiles stay null in JSON and NaN in feature arrays.
    receipt['features'] = {k: None if isinstance(v, float) and not np.isfinite(v) else v for k, v in features.items()}
    save_json(receipt_path, receipt)
    return cache_path, receipt

cache_paths, image_receipts, feature_rows = {}, [], []
for position, row in pretrial.iterrows():
    identity = [ARTICLE, ARCHIVE_SIZE, members[row.rgb_member].CRC, members[row.depth_member].CRC]
    path, receipt = cached_pair(row.sample_id, row.rgb_member, row.depth_member, read_member, identity)
    cache_paths[row.sample_id] = path
    image_receipts.append(dict(sample_id=row.sample_id, **receipt))
    feature_rows.append(receipt['features'])
    if (position + 1) % 25 == 0:
        print('Cached labeled pairs:', position + 1, '/', len(pretrial), flush=True)
features = pd.DataFrame(feature_rows)
assert len(features) == len(pretrial)
usable = ~features.very_dark & (features.depth_nonzero_fraction > 0)
train_idx = np.array([i for i in train_indices if usable.iloc[i]], dtype=int)
val_idx = np.array([i for i in validation_indices if usable.iloc[i]], dtype=int)
assert len(train_idx) >= 30 and len(val_idx) >= 10
for indices in [train_idx, val_idx]:
    assert (pretrial.loc[indices, TARGETS].notna().sum() >= 3).all()
features.assign(sample_id=pretrial.sample_id).to_parquet(ROOT / 'vision/pretrial_handcrafted_features.parquet', index=False)
save_json(ROOT / 'reports/source_receipts.json', image_receipts)
save_json(ROOT / 'reports/acquisition.json', dict(source_doi=f'10.4121/{ARTICLE}.v1', archive_size=ARCHIVE_SIZE,
    selected_pairs=len(pretrial), selected_compressed_bytes=selected_bytes,
    range_bytes_this_session=remote.transferred if remote else 0, whole_archive_md5_verified=False,
    verification='member CRC plus recorded SHA-256; validated session cache may be reused',
    publisher_test_read=False, infrared_read=False, usable_train=len(train_idx), usable_validation=len(val_idx)))
archive.close()
print('Ready:', len(train_idx), 'training pairs;', len(val_idx), 'validation pairs')

## Cell 6 — Original 04: simple vision baselines

Ridge fits a weighted sum of image measurements. Random Forest combines many small decision trees. Both learn from training plants only. MAE is average absolute error; RMSE gives larger mistakes more weight. Missing labels are skipped separately for each trait.

In [ ]:
FEATURE_NAMES = [c for c in features if c not in ['very_dark', 'empty_green_mask']]
X = features[FEATURE_NAMES].astype(float).to_numpy()
Y = pretrial[TARGETS].to_numpy(dtype=np.float32)
metric_rows, prediction_rows = [], []

def record_predictions(model_name, indices, predictions):
    """Score identical eligible validation rows for every method, trait by trait."""
    assert predictions.shape == (len(indices), len(TARGETS)) and np.isfinite(predictions).all()
    for j, target in enumerate(TARGETS):
        valid = np.isfinite(Y[indices, j])
        errors = predictions[valid, j] - Y[indices[valid], j]
        metric_rows.append(dict(model=model_name, target=target, unit=UNITS[j], n=int(valid.sum()),
                                mae=float(np.abs(errors).mean()), rmse=float(np.sqrt(np.square(errors).mean()))))
        for i, actual, predicted in zip(indices[valid], Y[indices[valid], j], predictions[valid, j]):
            prediction_rows.append(dict(sample_id=pretrial.loc[i, 'sample_id'], plant_group=pretrial.loc[i, 'plant_group'],
                                        model=model_name, target=target, actual=float(actual), predicted=float(predicted), split='validation'))

for name in ['train_mean', 'ridge', 'random_forest']:
    predictions = np.zeros((len(val_idx), len(TARGETS)))
    for j, target in enumerate(TARGETS):
        labeled = train_idx[np.isfinite(Y[train_idx, j])]
        if name == 'train_mean':
            predictions[:, j] = Y[labeled, j].mean()
            continue
        estimator = Ridge(alpha=10) if name == 'ridge' else RandomForestRegressor(
            n_estimators=300, min_samples_leaf=3, max_features=0.8, n_jobs=2, random_state=SEED)
        pipeline = make_pipeline(SimpleImputer(strategy='median', keep_empty_features=True), StandardScaler(), estimator)
        pipeline.fit(X[labeled], Y[labeled, j])
        predictions[:, j] = pipeline.predict(X[val_idx])
    record_predictions(name, val_idx, predictions)
save_json(ROOT / 'vision/vision_baseline.json', dict(features=FEATURE_NAMES, metrics=metric_rows,
    label_scope='2023 labeled single plants', segmentation_metrics='not available: no annotated segmentation masks',
    hyperparameters={'ridge_alpha': 10, 'forest_trees': 300, 'min_leaf': 3}))
display(pd.DataFrame(metric_rows))

## Cell 7 — Original 05: paired inputs and the RGB-D network

RGB-D means color plus depth. The RGB branch is MobileNetV3; a small convolutional network reads relative depth and its validity mask. The fused model joins both representations and the usable-depth fraction. A masked Huber loss learns only from labels that exist and reduces the effect of very large errors. A shared flip keeps color and depth lined up.

In [ ]:
class PlantDataset(Dataset):
    """Load a cached pair and apply one shared geometric augmentation."""
    def __init__(self, indices, mean, scale, augment=False):
        self.indices, self.mean, self.scale, self.augment = list(indices), mean, scale, augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        i = self.indices[position]
        with np.load(cache_paths[pretrial.loc[i, 'sample_id']]) as pair:
            rgb = pair['rgb'].astype(np.float32).transpose(2, 0, 1) / 255
            depth = pair['depth'].astype(np.float32)
        if self.augment and random.random() < 0.5:
            rgb, depth = rgb[:, :, ::-1].copy(), depth[:, :, ::-1].copy()
        # ImageNet's fixed normalization belongs to the pretrained model, not this dataset.
        rgb = (rgb - np.array([.485, .456, .406], dtype=np.float32)[:, None, None]) / np.array([.229, .224, .225], dtype=np.float32)[:, None, None]
        target = (Y[i] - self.mean) / self.scale
        mask = np.isfinite(target)
        return (torch.from_numpy(rgb.copy()), torch.from_numpy(depth.copy()),
                torch.tensor([features.loc[i, 'depth_nonzero_fraction']], dtype=torch.float32),
                torch.from_numpy(np.nan_to_num(target)), torch.from_numpy(mask))

class VisionNet(nn.Module):
    """Expose an embedding (compact image description) plus four trait predictions."""
    def __init__(self, mode='rgbd', pretrained=True):
        super().__init__()
        assert mode in ['rgb', 'depth', 'rgbd']
        self.mode = mode
        if mode != 'depth':
            weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
            self.rgb = mobilenet_v3_small(weights=weights).features
        if mode != 'rgb':
            self.depth = nn.Sequential(nn.Conv2d(2, 16, 5, stride=2, padding=2), nn.ReLU(),
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        dimension = (576 if mode != 'depth' else 0) + (64 if mode != 'rgb' else 0) + (1 if mode == 'rgbd' else 0)
        self.embedding = nn.Sequential(nn.Linear(dimension, 128), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(128, len(TARGETS)))

    def forward(self, rgb, depth, validity):
        parts = []
        if self.mode != 'depth':
            parts.append(self.rgb(rgb).mean(dim=(2, 3)))
        if self.mode != 'rgb':
            parts.append(self.depth(depth).flatten(1))
        if self.mode == 'rgbd':
            parts.append(validity)  # Keep the RGB-only comparison free of depth metadata.
        embedding = self.embedding(torch.cat(parts, dim=1))
        return self.head(embedding), embedding

def make_loader(indices, mean, scale, augment=False):
    """Single-process loading avoids notebook worker/pickling issues on Kaggle."""
    return DataLoader(PlantDataset(indices, mean, scale, augment), batch_size=BATCH_SIZE,
                      shuffle=augment, num_workers=0, pin_memory=True, drop_last=False)

def label_scaler(indices):
    """Fit each label's scale using only this model's training plants."""
    assert (np.isfinite(Y[indices]).sum(axis=0) >= 3).all()
    mean = np.nanmean(Y[indices], axis=0)
    scale = np.nanstd(Y[indices], axis=0)
    return mean.astype(np.float32), np.maximum(scale, 1.0).astype(np.float32)

def predict(model, indices, mean, scale):
    """Get deterministic predictions in the original measurement units."""
    model.eval()
    predictions, embeddings = [], []
    with torch.inference_mode():
        for rgb, depth, validity, _, _ in make_loader(indices, mean, scale):
            output, embedding = model(rgb.to(DEVICE), depth.to(DEVICE), validity.to(DEVICE))
            predictions.append(output.cpu().numpy() * scale + mean)
            embeddings.append(embedding.cpu().numpy())
    return np.concatenate(predictions), np.concatenate(embeddings)

## Cell 8 — Train RGB-only, depth-only, and RGB-D

An ablation removes one input to see what it contributes. All three models use the same plants, labels, and validation rows. Early stopping keeps the best validation checkpoint. Checkpoints are saved after each epoch so the same session can resume; do not change the data or settings mid-run.

In [ ]:
TRAINING_SPEC = dict(seed=SEED, size=IMAGE_SIZE, batch=BATCH_SIZE, max_epochs=MAX_EPOCHS,
    patience=PATIENCE, lr=LR, weight_decay=WEIGHT_DECAY, model='mobilenetv3small-depthcnn-128-v1',
    preprocess=PREPROCESS_VERSION, split_sha256=sha((ROOT / 'data/pretrial_split.parquet').read_bytes()),
    inputs=sha(json.dumps(image_receipts, sort_keys=True).encode()), versions=VERSIONS)

def fit_model(mode, fit_indices, monitor_indices, name, fixed_epochs=None):
    """Fit a model; OOF folds get a fixed epoch count and never see held-fold labels."""
    mean, scale = label_scaler(fit_indices)
    spec = dict(TRAINING_SPEC, mode=mode, fit_indices=list(map(int, fit_indices)),
                monitor_indices=list(map(int, monitor_indices)), fixed_epochs=fixed_epochs)
    fingerprint = sha(json.dumps(spec, sort_keys=True).encode())
    checkpoint = CACHE / f'{name}-{fingerprint[:16]}.pt'
    seed_all(SEED)
    model = VisionNet(mode).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_score, best_epoch, stale, start = float('inf'), 0, 0, 0
    best_state, history = None, []
    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)  # Only our own local checkpoint.
        assert saved['fingerprint'] == fingerprint
        model.load_state_dict(saved['state'])
        optimizer.load_state_dict(saved['optimizer'])
        for state in optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(DEVICE)
        best_state, best_score, best_epoch = saved['best_state'], saved['best_score'], saved['best_epoch']
        stale, start, history = saved['stale'], saved['epoch'], saved['history']
    limit = fixed_epochs or MAX_EPOCHS
    for epoch in range(start, limit):
        if fixed_epochs is None and stale >= PATIENCE:
            break
        seed_all(SEED + epoch)
        model.train()
        # Keep pretrained batch-normalization running statistics fixed on this small dataset.
        for layer in model.modules():
            if isinstance(layer, nn.BatchNorm2d):
                layer.eval()
        loss_sum, batches = 0.0, 0
        for rgb, depth, validity, target, mask in make_loader(fit_indices, mean, scale, augment=True):
            rgb, depth, validity = rgb.to(DEVICE), depth.to(DEVICE), validity.to(DEVICE)
            target, mask = target.to(DEVICE), mask.to(DEVICE)
            if not mask.any():
                continue
            optimizer.zero_grad(set_to_none=True)
            output, _ = model(rgb, depth, validity)
            loss = nn.functional.smooth_l1_loss(output[mask], target[mask])
            assert torch.isfinite(loss), 'Non-finite training loss'
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            loss_sum += loss.item()
            batches += 1
        assert batches
        if fixed_epochs is None:
            predictions, _ = predict(model, monitor_indices, mean, scale)
            per_target = np.nanmean(np.abs(predictions - Y[monitor_indices]) / scale, axis=0)
            score = float(per_target.mean())
            assert np.isfinite(score)
            improved = score < best_score
        else:
            # There is no fold-validation access here: preserve the last fixed-epoch state.
            score, improved = None, True
        if improved:
            best_score, best_epoch, stale = score, epoch + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        history.append(dict(epoch=epoch + 1, train_huber=loss_sum / batches, validation_scaled_mae=score))
        saved = dict(fingerprint=fingerprint, state=model.state_dict(), optimizer=optimizer.state_dict(),
                     best_state=best_state, best_score=best_score, best_epoch=best_epoch,
                     stale=stale, epoch=epoch + 1, history=history)
        temporary = checkpoint.with_suffix('.tmp')
        torch.save(saved, temporary)
        temporary.replace(checkpoint)
        print(name, 'epoch', epoch + 1, 'loss', round(loss_sum / batches, 4), 'validation', score, flush=True)
    assert best_state is not None
    model.load_state_dict(best_state)
    model.eval()
    info = dict(mode=mode, mean=mean.tolist(), scale=scale.tolist(), best_epoch=best_epoch,
                training_sample_ids=pretrial.loc[fit_indices, 'sample_id'].tolist(),
                training_plant_groups=sorted(pretrial.loc[fit_indices, 'plant_group'].unique().tolist()),
                spec=spec, history=history)
    save_json(ROOT / f'reports/{name}_training.json', info)
    return model, mean, scale, info

frozen_model = None
for mode in ['rgb', 'depth', 'rgbd']:
    model, mean, scale, info = fit_model(mode, train_idx, val_idx, mode)
    predictions, _ = predict(model, val_idx, mean, scale)
    record_predictions(mode, val_idx, predictions)
    if mode == 'rgbd':
        frozen_model, final_mean, final_scale, final_info = model, mean, scale, info
    else:
        del model
        gc.collect()
        torch.cuda.empty_cache()
metrics = pd.DataFrame(metric_rows)
assert metrics.groupby('target').n.nunique().eq(1).all(), 'Comparison rows differ.'
pd.DataFrame(prediction_rows).to_parquet(ROOT / 'vision/predictions.parquet', index=False)
save_json(ROOT / 'vision/metrics.json', dict(validation=metric_rows,
    protocol='plant-group-held-out development validation; not publisher test',
    segmentation='not scored: annotated masks unavailable',
    model_selection='RGB-D architecture fixed by plan; validation chooses epoch, not a claimed winner'))
display(metrics.pivot(index='target', columns='model', values='mae'))
# Freeze the exact model before feature extraction; its weights cannot learn from canopy images.
for parameter in frozen_model.parameters():
    parameter.requires_grad_(False)
torch.save({k: v.cpu() for k, v in frozen_model.state_dict().items()}, ROOT / 'models/vision.pt')
frozen_model.to(DEVICE)

## Cell 9 — Original 06: honest out-of-fold features

Each training plant gets predictions from a fold model trained on other plant groups. Fold models fit their own label scales. Their epoch count was chosen using the separate development validation set, never the held-fold labels. Fold embeddings have different coordinate systems, so only trait predictions are supplied as comparable OOF features.

In [ ]:
oof_rows, fold_audit = [], []
OOF_EPOCHS = int(final_info['best_epoch'])
for fold in range(N_FOLDS):
    held = np.array([i for i in train_idx if pretrial.loc[i, 'fold'] == fold], dtype=int)
    fit = np.array([i for i in train_idx if pretrial.loc[i, 'fold'] != fold], dtype=int)
    assert len(held) and len(fit)
    fit_groups, held_groups = set(pretrial.loc[fit, 'plant_group']), set(pretrial.loc[held, 'plant_group'])
    assert not fit_groups & held_groups
    fold_model, mean, scale, info = fit_model('rgbd', fit, [], f'oof_fold_{fold}', fixed_epochs=OOF_EPOCHS)
    predictions, _ = predict(fold_model, held, mean, scale)
    for position, i in enumerate(held):
        row = dict(sample_id=pretrial.loc[i, 'sample_id'], experiment_id='AGC4-pretrial-2023',
                   plant_group=pretrial.loc[i, 'plant_group'], fold=fold, split='train',
                   quality='usable_pair', feature_origin='group_out_of_fold')
        for j, target in enumerate(TARGETS):
            row['pred_' + target] = float(predictions[position, j])
        oof_rows.append(row)
    fold_audit.append(dict(fold=fold, fit_groups=sorted(fit_groups), held_groups=sorted(held_groups),
                           training_sample_ids=pretrial.loc[fit, 'sample_id'].tolist(),
                           held_sample_ids=pretrial.loc[held, 'sample_id'].tolist(), fixed_epochs=OOF_EPOCHS))
    del fold_model
    gc.collect()
    torch.cuda.empty_cache()
oof = pd.DataFrame(oof_rows)
assert oof.sample_id.is_unique and set(oof.sample_id) == set(pretrial.loc[train_idx, 'sample_id'])
assert np.isfinite(oof[['pred_' + t for t in TARGETS]].to_numpy()).all()
oof.to_parquet(ROOT / 'vision/oof_features.parquet', index=False)
save_json(ROOT / 'reports/oof_audit.json', fold_audit)
print('OOF rows:', len(oof), '| no held plant group used for that row’s model fit')

## Cell 10 — Frozen canopy features and uncertainty

The frozen encoder was trained on separate 2023 data, so 2024 fusion training may use it without learning from 2024 labels. An embedding is a compact image description. Repeated dropout predictions show model variability; their variance is not a calibrated confidence interval. Dark or unusable frames keep their IDs and quality flags, but their model features stay missing.

In [ ]:
def pair_tensors(path, validity):
    """Use exactly the training preprocessing for a new cached canopy pair."""
    with np.load(path) as pair:
        rgb = pair['rgb'].astype(np.float32).transpose(2, 0, 1) / 255
        depth = pair['depth'].astype(np.float32)
    rgb = (rgb - np.array([.485, .456, .406], dtype=np.float32)[:, None, None]) / np.array([.229, .224, .225], dtype=np.float32)[:, None, None]
    return (torch.from_numpy(rgb).unsqueeze(0).to(DEVICE), torch.from_numpy(depth).unsqueeze(0).to(DEVICE),
            torch.tensor([[validity]], dtype=torch.float32, device=DEVICE))

canopy_rows, canopy_receipts = [], []
model_hash = sha((ROOT / 'models/vision.pt').read_bytes())
spatial_by_id = spatial.set_index('image_id')
seed_all(SEED)
for row in images.itertuples(index=False):
    path, receipt = cached_pair(row.image_id, row.rgb_path, row.depth_path, s02.read, sha(s02.manifest_bytes))
    assert receipt['rgb_sha256'] == row.rgb_sha256 and receipt['depth_sha256'] == row.depth_sha256
    quality = receipt['features']
    usable_pair = not quality['very_dark'] and quality['depth_nonzero_fraction'] > 0
    prior = spatial_by_id.loc[row.image_id]
    result = dict(image_id=row.image_id, experiment_id=row.experiment_id, compartment_id=str(row.compartment_id),
        camera_id=row.camera_id, ts_utc=row.ts_utc, split=row.split, chronological_split=row.chronological_split,
        valid_depth_fraction=quality['depth_nonzero_fraction'], validity_rule='provisional nonzero native codes',
        qc_flags=str(prior.qc_flags), green_mask_fraction=float(prior.mask_pixel_fraction),
        usable_model_feature=usable_pair, transfer_validated=False, metric_geometry_verified=False,
        depth_unit='unverified_native', feature_origin='frozen_encoder_disjoint_2023', model_sha256=model_hash)
    result.update({'embedding_' + str(j).zfill(3): np.nan for j in range(128)})
    for target in TARGETS:
        result['proxy_' + target + '_mean'] = np.nan
        result['proxy_' + target + '_variance'] = np.nan
    if usable_pair:
        inputs = pair_tensors(path, quality['depth_nonzero_fraction'])
        frozen_model.eval()
        with torch.inference_mode():
            _, embedding = frozen_model(*inputs)
            result.update({'embedding_' + str(j).zfill(3): float(v) for j, v in enumerate(embedding[0].cpu().numpy())})
            # Enable only dropout; batch-normalization statistics remain frozen.
            for layer in frozen_model.modules():
                if isinstance(layer, nn.Dropout):
                    layer.train()
            draws = np.stack([frozen_model(*inputs)[0][0].cpu().numpy() * final_scale + final_mean for _ in range(MC_PASSES)])
            for j, target in enumerate(TARGETS):
                result['proxy_' + target + '_mean'] = float(draws[:, j].mean())
                result['proxy_' + target + '_variance'] = float(draws[:, j].var(ddof=1))
        frozen_model.eval()
    canopy_rows.append(result)
    canopy_receipts.append(dict(image_id=row.image_id, **receipt))
vision = pd.DataFrame(canopy_rows)
assert vision.image_id.is_unique and set(vision.image_id) == set(images.image_id)
assert not vision.transfer_validated.any()
assert sha((ROOT / 'models/vision.pt').read_bytes()) == model_hash
vision.to_parquet(ROOT / 'vision/vision_features.parquet', index=False)
save_json(ROOT / 'reports/canopy_receipts.json', canopy_receipts)
vision_config = dict(code_version='1.0', mode='rgbd', embedding_size=128, targets=TARGETS, units=UNITS,
    label_mean=final_mean.tolist(), label_scale=final_scale.tolist(), training=TRAINING_SPEC,
    input_size=IMAGE_SIZE, rgb_normalization='ImageNet', resize='aspect-preserving paired letterbox',
    depth='nearest resize; nonzero depth / image median, clipped [0,4]; second channel nonzero mask',
    mc_dropout_passes=MC_PASSES, uncertainty='dropout variance only; not calibrated predictive coverage',
    model_sha256=model_hash, encoder_training_experiment='AGC4-pretrial-2023',
    canopy_trait_status='unvalidated single-plant-to-canopy transfer; proxy fields only',
    fusion_contract='2024 features use a frozen encoder trained on disjoint data; 2023 OOF traits are in a separate table',
    next='EDEN-07: backward time join, image-age/missingness masks, training-only scaling; never forward-fill future images')
save_json(ROOT / 'models/vision_config.json', vision_config)
display(vision.groupby('split').agg(images=('image_id', 'size'), usable=('usable_model_feature', 'sum')))

## Cell 11 — Review results and package the handoff

The checks below verify the original 04–06 deliverables. They do not require the neural model to beat the baselines. Read the validation table before deciding whether the features are useful. The next planned notebook is 07, alignment; geometry and canopy-transfer limitations remain visible.

In [ ]:
expected_models = {'train_mean', 'ridge', 'random_forest', 'rgb', 'depth', 'rgbd'}
assert set(metrics.model) == expected_models and len(metrics) == len(expected_models) * len(TARGETS)
assert np.isfinite(metrics[['mae', 'rmse']].to_numpy()).all()
assert (metrics.n >= 3).all()
embedding_columns = [c for c in vision if c.startswith('embedding_')]
proxy_columns = [c for c in vision if c.startswith('proxy_')]
good = vision.usable_model_feature
assert good.any() and np.isfinite(vision.loc[good, embedding_columns + proxy_columns].to_numpy()).all()
assert vision.loc[~good, embedding_columns + proxy_columns].isna().all().all()
assert (vision.loc[good, [c for c in proxy_columns if c.endswith('_variance')]] >= 0).all().all()
assert set(vision.compartment_id) == set(images.compartment_id)
for column in ['split', 'chronological_split']:
    assert vision.set_index('image_id')[column].equals(images.set_index('image_id')[column])
assert all(not set(f['fit_groups']) & set(f['held_groups']) for f in fold_audit)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, target in zip(axes, TARGETS):
    table = metrics[metrics.target == target]
    axis.bar(table.model, table.mae)
    axis.set_title(target)
    axis.tick_params(axis='x', labelrotation=65)
    axis.set_ylabel('validation MAE')
fig.tight_layout()
fig.savefig(ROOT / 'reports/validation_comparison.png', dpi=120)
plt.show()
plt.close(fig)

status = 'PASS_VISION_PIPELINE_TRANSFER_UNVALIDATED'
acceptance = dict(status=status, original_04='handcrafted features, Ridge/forest baselines, grouped validation',
    original_05='trained RGB/depth/RGB-D comparisons with masked trait supervision and paired augmentation',
    original_06='frozen encoder, canopy embeddings/proxy means/variance/quality, genuine grouped pre-trial OOF traits',
    performance_acceptance='not asserted; inspect validation metrics and domain transfer before using as trusted traits',
    canopy_images=len(vision), usable_canopy_images=int(good.sum()), oof_rows=len(oof),
    limitations=['Single-plant labels do not validate canopy traits.', 'No metric depth geometry.',
                 'Color masks need review; no segmentation ground truth.', 'Dropout variance is uncalibrated.',
                 'Small 36-pair canopy pilot; no dense temporal vision coverage.',
                 'Publisher test excluded from notebook; test metadata was inspected during preparation.'],
    next_notebook='EDEN-07 Alignment', original_plan_endpoint='implemented; execution and scientific usefulness require result review')
save_json(ROOT / 'reports/acceptance.json', acceptance)
(ROOT / 'LICENSES.txt').write_text(
    '2023 pre-trial and 2024 challenge datasets: CC BY 4.0, Wageningen University & Research / 4TU.ResearchData.\n'
    f'Pre-trial DOI: 10.4121/{ARTICLE}.v1\n2024 DOI: 10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1\n'
    'Torchvision MobileNetV3 weights and code retain their upstream attribution. No raw images included.\n', encoding='utf-8')

# Keep an explicit allow-list: old files from an interrupted run do not enter the handoff.
required = ['data/splits.json', 'data/pretrial_split.parquet', 'vision/vision_baseline.json',
    'vision/metrics.json', 'vision/predictions.parquet', 'vision/pretrial_handcrafted_features.parquet',
    'vision/oof_features.parquet', 'vision/vision_features.parquet', 'models/vision.pt', 'models/vision_config.json',
    'reports/runtime.json', 'reports/input_verification.json', 'reports/pretrial_ReadMe.md',
    'reports/pretrial_camera_config.json', 'reports/label_contract.json', 'reports/source_receipts.json',
    'reports/acquisition.json', 'reports/rgb_training.json', 'reports/depth_training.json',
    'reports/rgbd_training.json', 'reports/oof_audit.json', 'reports/canopy_receipts.json',
    'reports/validation_comparison.png', 'reports/acceptance.json', 'LICENSES.txt']
required += [f'reports/oof_fold_{i}_training.json' for i in range(N_FOLDS)]
files = []
for relative in required:
    payload = (ROOT / relative).read_bytes()
    files.append(dict(path=relative, size=len(payload), sha256=sha(payload)))
manifest = dict(schema_version='1.0', producer='EDEN-06', code_version='1.0', status=status,
    original_modules_covered=['04 vision baselines', '05 RGB-D model', '06 feature extraction'],
    generated_utc=datetime.now(timezone.utc).isoformat(), upstream=verified, files=files,
    model_sha256=model_hash, test_scored=False, metric_geometry_created=False, canopy_transfer_validated=False)
save_json(ROOT / 'manifest.json', manifest)
with zipfile.ZipFile(RESULT, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as bundle:
    for relative in required + ['manifest.json']:
        bundle.write(ROOT / relative, 'eden/' + relative)
with zipfile.ZipFile(RESULT) as bundle:
    assert bundle.testzip() is None
    for item in files:
        payload = bundle.read('eden/' + item['path'])
        assert len(payload) == item['size'] and sha(payload) == item['sha256']
print(status)
print('Return this file for review:', RESULT, '| bytes:', RESULT.stat().st_size)

### If a cell stops

- **Missing EDEN input:** attach the 01, 02 and 03 result bundles. ZIP and extracted forms work; different versions of the same producer are rejected.
- **Internet/range error:** keep Internet on and rerun from Cell 3. Completed resized pairs in this session are checked and reused. A server that ignores partial requests is deliberately rejected.
- **GPU memory error:** restart the session and reduce `BATCH_SIZE` to 8 before running. Settings are included in checkpoint identities.
- **Changed source schema or insufficient labels:** return the complete error; do not remove the check or invent labels.
- **Poor validation scores:** the notebook still records the comparison honestly. Share the result ZIP so we can decide whether these features are useful before Notebook 07.

**Next:** download `EDEN-06-Vision-Recovery-and-Features-Results-v1.0.zip`. You do not need to upload the image cache. Notebook 07 will use EDEN-01 plus this result to align image features with climate timestamps.